In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.read.format("delta").load(silver_path + "/iot")

# Partition by device_id and event_date, order by timestamp ascending/descending
window_spec_asc = Window.partitionBy("device_id", "event_date").orderBy("timestamp")
window_spec_desc = Window.partitionBy("device_id", "event_date").orderBy(F.col("timestamp").desc())

# First and last event for each device/day
first_event_df = silver_df.withColumn("rn", F.row_number().over(window_spec_asc)) \
                          .filter(F.col("rn") == 1) \
                          .drop("rn")

last_event_df = silver_df.withColumn("rn", F.row_number().over(window_spec_desc)) \
                         .filter(F.col("rn") == 1) \
                         .drop("rn")

# Join and calculate expected readings
result_df = first_event_df.alias("f").join(
    last_event_df.alias("l"),
    ["device_id", "event_date"]
).select(
    "device_id",
    "event_date",
    ((F.unix_timestamp("l.timestamp") - F.unix_timestamp("f.timestamp")) / 5).alias("expected_readings"),
    "f.is_faulty"
)


actual_counts_df = silver_df.groupBy("device_id", "event_date") \
    .agg(F.count("*").alias("actual_count"))

# Join with result_df (which should have expected_readings already)
final_df = actual_counts_df.join(result_df, ["device_id", "event_date"]) \
    .withColumn(
        "uptime_percent",
        round(F.when(F.col("expected_readings") > 0,
               (F.col("actual_count") / F.col("expected_readings")) * 100
        ).otherwise(F.lit(0)), 2))
    
actual_counts_df = silver_df.groupBy("device_id", "event_date") \
    .agg(F.count("*").alias("actual_count"))

# Join with result_df (which should have expected_readings already)
final_df = actual_counts_df.join(result_df, ["device_id", "event_date"]) \
    .withColumn(
        "uptime_percent",
        round(F.when(F.col("expected_readings") > 0,
               (F.col("actual_count") / F.col("expected_readings")) * 100
        ).otherwise(F.lit(0)), 2))

count_df= silver_df.filter(col("is_faulty") == True).withColumn(
    "faultCount",
    count("*").over(Window.partitionBy("device_id", "event_date"))
)

w = Window.partitionBy("device_id").orderBy("timestamp")

fault_df_t = silver_df.filter(F.col("is_faulty") == True) \
    .withColumn("prev_fault", F.lag("timestamp").over(w))

fault_df_t = fault_df_t.withColumn(
    "diff_seconds",
    F.unix_timestamp("timestamp") - F.unix_timestamp("prev_fault")
)

mtbf_df = fault_df_t.groupBy("device_id") \
    .agg(F.avg("diff_seconds").alias("mtbf_seconds"))
mtbf_df.show()

gold_path = "s3://my-iot-data-bucket-2025/gold"
res_df = final_df.join(mtbf_df, "device_id", "left").join(count_df, "device_id", "left") \
    .select(
        mtbf_df["mtbf_seconds"],
        count_df["faultCount"],
        *[final_df[col] for col in final_df.columns if col != "is_faulty"],
    )
display(res_df)
res_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy("event_date").save(gold_path + "/iot")